# 94 — Campaign B staged M6 batch (4 packages)

義務クローズ済み + `M5_COMPLETE` の候補を **live_parent M6**（notebook 70 相当）でまとめて回す。

- `mode=live_parent`（**81 production paperspace gate ではない**）
- 親は child `M5-*` + `artifacts/one_step_certificate`
- 結果は `CERTIFIED` または `NOT_CERTIFIED`（後者は majorant が q_cert&lt;1 を示せなかった検証済み失敗）
- Campaign B の `claim_scope` は `SCREENING_ONLY` のまま
- 連続極限・質量ギャップは主張しない


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'm6_batch.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/m6_batch.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.m6_batch import list_m6_queue, run_m6_batch

validate_persistent_root()
MAX_PACKAGES = 4
MAX_QUEUE = 50
ONLY_CAMPAIGN = None  # e.g. 'M7-20260720T152525Z-b-d87cf59a77dc'

QUEUE = list_m6_queue(
    PERSIST_ROOT,
    max_candidates=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'queue_size': len(QUEUE),
    'top': [
        {
            'candidate_id': r.get('candidate_id'),
            'q_upper': r.get('q_upper'),
            'm5_run_id': r.get('m5_run_id'),
            'm6_run_id': r.get('m6_run_id'),
            'm5_certification_status': r.get('m5_certification_status'),
            'gate_status': r.get('gate_status'),
        }
        for r in QUEUE[:10]
    ],
}, indent=2, ensure_ascii=False, default=str))


In [ ]:
SESSION = run_m6_batch(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    max_packages=MAX_PACKAGES,
    max_queue=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'queue_size': SESSION.get('queue_size'),
    'attempted': SESSION.get('attempted'),
    'm6_complete': SESSION.get('m6_complete'),
    'm6_certified_count': SESSION.get('m6_certified_count'),
    'm6_not_certified_count': SESSION.get('m6_not_certified_count'),
    'errors': SESSION.get('errors'),
    'results': SESSION.get('results'),
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_m6' / 'LATEST_M6_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))
